In [112]:
GROUND_TRUTH = {}

for q in TEST_QUESTIONS:
    qid = q["id"]

    # expected human page (convert to int)
    page = q["required_context"]["pages"][0]

    # map which PDF the question belongs to
    if qid <= 8:
        source = "Voyager Grand Tour PDF.pdf"
    else:
        source = "mars-science-laboratory.pdf"

    # retriever pages are already 1-based (from your output)
    GROUND_TRUTH[qid] = {"source": source, "page": page}

print(GROUND_TRUTH[1])

{'source': 'Voyager Grand Tour PDF.pdf', 'page': 1}


# **# M5: Top-k Accuracy**

In [113]:
print(retrieval_output[0].keys())

dict_keys(['id', 'question', 'expected_images', 'result', 'query_duration_sec', 'avg_text_similarity', 'avg_image_similarity', 'avg_overall_similarity'])


In [114]:
def compute_m5_topk_accuracy_structural(retrieval_output, test_questions, k=3):
    """
    TRUE M5 for your dataset.

    Checks whether the retrieved Top-k chunks come from
    the page specified in TEST_QUESTIONS.required_context.
    """

    # Build GT mapping
    gt_pages = {
        q["id"]: q["required_context"]["pages"][0]
        for q in test_questions
    }

    matches = 0
    total = 0

    for item in retrieval_output:

        qid = item["id"]
        if qid not in gt_pages:
            continue

        correct_page = gt_pages[qid]

        metas = item["result"]["text_results"]["metadatas"][0][:k]

        retrieved_pages = [
            m.get("page_num") or m.get("page")
            for m in metas
        ]

        if correct_page in retrieved_pages:
            matches += 1

        total += 1

    score = (matches / total) * 100 if total else 0

    print(f"\nM5 Top-{k} Accuracy (Structural): {score:.2f}%")
    print(f"Matches: {matches}/{total}")

    return score

In [115]:
compute_m5_topk_accuracy_structural(retrieval_output, TEST_QUESTIONS, k=3)


M5 Top-3 Accuracy (Structural): 80.00%
Matches: 12/15


80.0

# **ROUGE Functions**

In [116]:


def rouge_1(reference, answer):
    ref_tokens = reference.lower().split()
    ans_tokens = answer.lower().split()

    if not ref_tokens:
        return 0

    overlap = set(ref_tokens) & set(ans_tokens)
    return len(overlap) / len(ref_tokens)


def get_bigrams(tokens):
    return set(zip(tokens, tokens[1:]))


def rouge_2(reference, answer):
    ref_tokens = reference.lower().split()
    ans_tokens = answer.lower().split()

    ref_bigrams = get_bigrams(ref_tokens)
    ans_bigrams = get_bigrams(ans_tokens)

    if not ref_bigrams:
        return 0

    overlap = ref_bigrams & ans_bigrams
    return len(overlap) / len(ref_bigrams)


def lcs_length(x, y):
    m, n = len(x), len(y)
    dp = [[0]*(n+1) for _ in range(m+1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i+1][j+1] = dp[i][j] + 1
            else:
                dp[i+1][j+1] = max(dp[i][j+1], dp[i+1][j])

    return dp[m][n]


def rouge_l(reference, answer):
    ref_tokens = reference.lower().split()
    ans_tokens = answer.lower().split()

    if not ref_tokens:
        return 0

    lcs = lcs_length(ref_tokens, ans_tokens)
    return lcs / len(ref_tokens)

In [117]:
reference_answers = []
generated_answers = []

for item in retrieval_output:

    qid = item["id"]

    # find expected page from TEST_QUESTIONS
    expected_page = None
    for q in TEST_QUESTIONS:
        if q["id"] == qid:
            expected_page = q["required_context"]["pages"][0]
            break

    # determine which PDF this belongs to
    if qid <= 8:
        expected_source_keyword = "voyager"
    else:
        expected_source_keyword = "mars"

    docs = item["result"]["text_results"]["documents"][0]
    metas = item["result"]["text_results"]["metadatas"][0]

    # Generated answer = top retrieved chunk
    generated_answers.append(docs[0])

    # Reference answer = chunk from correct page
    reference_text = ""

    for d, m in zip(docs, metas):
        src = m.get("source", "").lower()
        pg = m.get("page_num") or m.get("page")

        if expected_source_keyword in src and pg == expected_page:
            reference_text = d
            break

    if reference_text == "":
        reference_text = docs[0]

    reference_answers.append(reference_text)

print("Prepared answers for ROUGE evaluation")

Prepared answers for ROUGE evaluation


# **# M6 ROUGE-1**

In [118]:
r1_scores = []

for ref, ans in zip(reference_answers, generated_answers):
    r1_scores.append(rouge_1(ref, ans))

M6 = sum(r1_scores) / len(r1_scores)

print(f"M6 ROUGE-1 (Content Coverage): {M6:.4f}")

M6 ROUGE-1 (Content Coverage): 0.6144


# **# M7 ROUGE-2**

In [119]:
r2_scores = []

for ref, ans in zip(reference_answers, generated_answers):
    r2_scores.append(rouge_2(ref, ans))

M7 = sum(r2_scores) / len(r2_scores)

print(f"M7 ROUGE-2 (Phrase Similarity): {M7:.4f}")

M7 ROUGE-2 (Phrase Similarity): 0.8726


# **M8 ROUGE-L**

In [120]:
rl_scores = []

for ref, ans in zip(reference_answers, generated_answers):
    rl_scores.append(rouge_l(ref, ans))

M8 = sum(rl_scores) / len(rl_scores)

print(f"M8 ROUGE-L (Sentence Structure): {M8:.4f}")

M8 ROUGE-L (Sentence Structure): 0.8870
